# NISAR над Каховським водосховищем: часовий ряд червень–листопад 2026

Продовження `nisar_kakhovka_research_v2.ipynb`. Там дві дати показали, що сцени забиті радіоперешкодами (RFI), і порівняння двох знімків не відрізняє реальних змін від шуму. Тут беремо **всі знімки з `Project/data/h5`**:
* **143_A_025**: висхідний прохід о ~02:39 UTC, до 11 дат 24.06–03.11;
* **108_D_065**: низхідний прохід о ~16:46 UTC, 11 дат 21.06–31.10.

Два треки з різною геометрією й різним часом доби — головний інструмент перевірки. **Справжня зміна поверхні має проявитися в обох, а перешкода — ні.**

**План**
1. Інвентар і кеш: з кожного файлу читаємо лише вікно AOI (обидві смуги, сітка 80 м).
2. Якість кожної дати: NESZ, rfiLikelihood і **рівень перешкод, виміряний на воді**.
3. Мапи всіх дат і карта «фону перешкод» для кожної дати; експериментальна корекція.
4. Багатодатні композити: що на ділянці стабільне, а що змінюється за сезон.
5. Часові ряди за типами поверхні проти погоди (ERA5).
6. Вода: NISAR проти DSWx-S1 на кожну дату з навчанням на інших датах.
7. Сезонні зміни: збирання врожаю на ріллі та узгодженість двох треків.
8. Стійкі зміни «початок сезону → кінець сезону» і їхня перевірка другим треком.

Район — зона бойових дій. Аналіз описує довкілля агреговано і **не** інтерпретує джерела перешкод.

## 0. Налаштування

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import os, re, json, warnings, urllib.request, urllib.parse

os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import h5py
from scipy.ndimage import median_filter, uniform_filter, zoom
from scipy.interpolate import RegularGridInterpolator
from scipy.stats import mannwhitneyu, pearsonr
from pyproj import Transformer
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from IPython.display import display

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 9})

DATA_DIR = Path('..') / 'data'
if not (DATA_DIR / 'h5').exists():
    DATA_DIR = next(p / 'data' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'h5').exists())
PROJECT = DATA_DIR.parent
TRACKS = ['143_A_025', '108_D_065']      # перший — основний
MAIN, ALT = TRACKS
BBOX = (33.4, 46.9, 35.2, 47.8)
EPSG, RES = 32636, 80.0
CACHE = DATA_DIR / 'cache_ts'; CACHE.mkdir(exist_ok=True)
OUT = DATA_DIR / 'analysis_timeseries'; OUT.mkdir(exist_ok=True)
GRID = '/science/LSAR/GCOV/grids'
CAL = '/science/LSAR/GCOV/metadata/calibrationInformation'
PAR = '/science/LSAR/GCOV/metadata/processingInformation/parameters'
db = lambda a: 10 * np.log10(a)

# Рівень обробки з data_plan (у назві файлу його немає)
LEVEL, PLAN_MB = {}, {}
for f in (PROJECT / 'data_plan').glob('kakhovka_urls_*.txt'):
    for line in f.read_text(encoding='utf-8').splitlines():
        if line.count('\t') == 4:
            _, lev, _, size, url = line.split('\t'); name = url.rsplit('/', 1)[-1]
            LEVEL[name], PLAN_MB[name] = lev, float(size.rstrip('MB'))

def complete(p):
    """Відкидає недокачані файли: замалий розмір або HDF5 не відкривається."""
    if p.name in PLAN_MB and p.stat().st_size / 2**20 < 0.97 * PLAN_MB[p.name]:
        return False
    try:
        with h5py.File(p, 'r') as f:
            return '/science/LSAR/GCOV/grids/frequencyB/HHHH' in f
    except OSError:
        return False

rows = []
for p in sorted((DATA_DIR / 'h5').glob('NISAR_L2_*GCOV_*.h5')):
    m = re.search(r'_(\d{3}_[AD]_\d{3})_\d{4}_(\w{4})_\w_(\d{8}T\d{6})_\d{8}T\d{6}_(P\d{5})_', p.name)
    if m and m.group(1) in TRACKS:
        if not complete(p):
            print('ПРОПУСКАЮ (недокачаний або пошкоджений):', p.name); continue
        rows.append({'трек': m.group(1), 'час': datetime.strptime(m.group(3), '%Y%m%dT%H%M%S'), 'реліз': m.group(4),
                     'рівень': LEVEL.get(p.name, '?'), 'ГБ': round(p.stat().st_size / 2**30, 2), 'path': p})
SCENES = pd.DataFrame(rows).sort_values(['трек', 'час']).reset_index(drop=True)
SCENES['дата'] = SCENES['час'].dt.strftime('%Y-%m-%d')
display(SCENES.drop(columns='path').groupby('трек').agg(дат=('дата', 'size'), перша=('дата', 'first'), остання=('дата', 'last'), ГБ=('ГБ', 'sum')))
for t in TRACKS:
    plan = sorted(datetime.strptime(re.search(r'_(\d{8})T', k).group(1), '%Y%m%d').strftime('%Y-%m-%d') for k in LEVEL if f'_{t}_' in k)
    have = set(SCENES.loc[SCENES['трек'] == t, 'дата'])
    print(f'{t}: є {len(have)} з {len(plan)} дат у плані; бракує: {[d for d in plan if d not in have] or "нічого"}')

# Спільна сітка 80 м: усі кадри NISAR GCOV лежать на одній глобальній решітці (центри на 40 м mod 80)
_t = Transformer.from_crs(4326, EPSG, always_xy=True)
_X, _Y = _t.transform([BBOX[0], BBOX[0], BBOX[2], BBOX[2]], [BBOX[1], BBOX[3], BBOX[1], BBOX[3]])
X = np.arange(np.floor((min(_X) - 40) / 80) * 80 + 40, max(_X), RES)
Y = np.arange(np.ceil((max(_Y) - 40) / 80) * 80 + 40, min(_Y), -RES)
H, W = len(Y), len(X)
TRANSFORM = from_origin(X[0] - RES / 2, Y[0] + RES / 2, RES, RES)
EXTENT = [(X[0] - 40) / 1e3, (X[-1] + 40) / 1e3, (Y[-1] - 40) / 1e3, (Y[0] + 40) / 1e3]
PIX_KM2 = RES**2 / 1e6
print(f'Сітка AOI: {H}×{W} пікселів по 80 м')

## 1. Читання вікна AOI з кожного файлу (з кешем)

Перший запуск займає ~15 с на файл, бо frequencyA (20 м) усереднюється 4×4 у лінійній потужності до 80 м. Далі все береться з `data/cache_ts`.

In [ ]:
def _clip(i0, n, N):
    a, b = max(i0, 0), min(i0 + n, N)
    return slice(a, b), slice(a - i0, b - i0)

def extract(path):
    out = {}
    with h5py.File(path, 'r') as f:
        meta = {'time': f['/science/LSAR/identification/zeroDopplerStartTime'][()].decode()[:19],
                'version': f['/science/LSAR/identification/productVersion'][()].decode()}
        meta |= {k: f[f'{PAR}/{k}'][()].decode() for k in ('noiseCorrectionApplied', 'rfiMitigationApplied')}
        for fr in ('frequencyB', 'frequencyA'):
            g = f[f'{GRID}/{fr}']; xs, ys = g['xCoordinates'][:], g['yCoordinates'][:]
            arr = {c: np.full((H, W), np.nan, 'f4') for c in ('HHHH', 'HVHV')}
            if fr == 'frequencyB':
                c0 = int(round((X[0] - xs[0]) / 80)); r0 = int(round((ys[0] - Y[0]) / 80))
                assert abs((X[0] - xs[0]) / 80 - c0) < 1e-3 and abs((ys[0] - Y[0]) / 80 - r0) < 1e-3, 'сітка не збігається'
                (sr, dr), (sc, dc) = _clip(r0, H, len(ys)), _clip(c0, W, len(xs))
                m = g['mask'][sr, sc]; ok = (m > 0) & (m < 255)
                for c in arr:
                    v = g[c][sr, sc].astype('f4'); arr[c][dr, dc] = np.where(ok & (v > 0), db(np.maximum(v, 1e-12)), np.nan)
            else:
                a0 = int(round((X[0] - 30 - xs[0]) / 20)); b0 = int(round((ys[0] - (Y[0] + 30)) / 20))
                kc0, kc1 = (-(a0 // 4) if a0 < 0 else 0), min(W, (len(xs) - a0) // 4)
                kr0, kr1 = (-(b0 // 4) if b0 < 0 else 0), min(H, (len(ys) - b0) // 4)
                cs = slice(a0 + 4 * kc0, a0 + 4 * kc1); w_ = kc1 - kc0
                for i in range(kr0, kr1, 128):
                    h = min(128, kr1 - i); rs = slice(b0 + 4 * i, b0 + 4 * (i + h))
                    ok = ((lambda m: (m > 0) & (m < 255))(g['mask'][rs, cs])).reshape(h, 4, w_, 4).all(axis=(1, 3))
                    for c in arr:
                        v = np.nanmean(g[c][rs, cs].astype('f4').reshape(h, 4, w_, 4), axis=(1, 3))
                        arr[c][i:i + h, kc0:kc1] = np.where(ok & (v > 0), db(np.maximum(v, 1e-12)), np.nan)
            tag = fr[-1]
            out[f'hh{tag}'], out[f'hv{tag}'] = arr['HHHH'].astype('f2'), arr['HVHV'].astype('f2')
            for pol in ('HH', 'HV'):
                nes = f[f'{CAL}/{fr}/noiseEquivalentBackscatter']
                a = db(np.where(nes[pol][:] > 0, nes[pol][:], np.nan))
                ip = RegularGridInterpolator((nes['yCoordinates'][:][::-1], nes['xCoordinates'][:]), a[::-1], bounds_error=False, fill_value=np.nan)
                YY, XX = np.meshgrid(Y, X, indexing='ij')
                out[f'nesz_{pol.lower()}{tag}'] = ip((YY, XX)).astype('f2')
                meta[f'rfi_{pol}{tag}'] = float(f[f'{CAL}/{fr}/{pol}/rfiLikelihood'][()])
    out['meta'] = np.array(json.dumps(meta))
    return out

STACK = {}
for t in TRACKS:
    sub = SCENES[SCENES['трек'] == t]
    S = {k: [] for k in ('hhB', 'hvB', 'hhA', 'hvA', 'nesz_hhB', 'nesz_hvB', 'nesz_hhA', 'nesz_hvA')}; metas = []
    for _, r in sub.iterrows():
        cf = CACHE / f'{t}_{r["час"]:%Y%m%dT%H%M}.npz'
        if not cf.exists():
            print('читаю', r['path'].name[:70]); np.savez(cf, **extract(r['path']))
        z = np.load(cf)
        for k in S: S[k].append(z[k])
        metas.append(json.loads(str(z['meta'])))
    STACK[t] = {k: np.stack(v).astype('f4' if k in ('hhB', 'hvB') else 'f2') for k, v in S.items()} | {'meta': metas, 'time': list(sub['час']), 'scenes': sub}
    print(t, 'дат:', len(metas), 'масив', STACK[t]['hhB'].shape)

## 2. Довідкові шари: WorldCover, JRC (дно водосховища), DSWx-S1 на кожну дату

In [ ]:
def to_grid(src_path, resampling=Resampling.nearest, nodata=255):
    dst = np.full((H, W), nodata, 'u1')
    with rasterio.open(src_path) as src:
        reproject(rasterio.band(src, 1), dst, dst_transform=TRANSFORM, dst_crs=f'EPSG:{EPSG}',
                  src_nodata=src.nodata if src.nodata is not None else nodata, dst_nodata=nodata, resampling=resampling)
    return dst

WC = to_grid(DATA_DIR / 'aux' / 'worldcover_2021_aoi.tif', Resampling.mode, nodata=0)
JRC = to_grid(DATA_DIR / 'aux' / 'jrc_occurrence_aoi.tif', Resampling.average, nodata=255).astype('f4')
JRC[JRC > 100] = np.nan
RES_BED = JRC >= 50
LAND_HIST = (np.nan_to_num(JRC, nan=0) < 5) & (WC != 80)        # суша за історією, без міток поточної дати
WC_NAMES = {10: 'дерева', 20: 'чагарники', 30: 'трави', 40: 'рілля', 50: 'забудова', 80: 'вода (2021)', 90: 'водно-болотні'}

S1_FILES = []
for p in (DATA_DIR / 'opera_dswx' / 's1').glob('*_B01_WTR.tif'):
    S1_FILES.append((datetime.strptime(re.search(r'_T\w{5}_(\d{8}T\d{6})Z_', p.name).group(1), '%Y%m%dT%H%M%S'), p))

def dswx_for(t0, max_days=3.0):
    cf = CACHE / f'dswx_s1_{t0:%Y%m%dT%H%M}.npz'
    if cf.exists():
        z = np.load(cf); return z['mos'], float(z['dt'])
    mos = np.full((H, W), 255, 'u1'); dts = []
    for t, p in sorted(S1_FILES, key=lambda tp: abs((tp[0] - t0).total_seconds())):
        dt = abs((t - t0).total_seconds()) / 86400
        if dt > max_days: break
        a = to_grid(p, Resampling.mode); fill = (mos >= 250) & (a < 250)
        if fill.any(): mos[fill] = a[fill]; dts.append(dt * fill.sum())
    mdt = sum(dts) / max((mos < 250).sum(), 1)
    np.savez_compressed(cf, mos=mos, dt=mdt)
    return mos, mdt

for t in TRACKS:
    ref = [dswx_for(tt) for tt in STACK[t]['time']]
    STACK[t]['dswx'] = np.stack([r[0] for r in ref]); STACK[t]['dswx_dt'] = [r[1] for r in ref]
print('Дно водосховища (JRC ≥ 50 %):', round(RES_BED.sum() * PIX_KM2), 'км²')

## 3. Якість кожної дати: «зонд перешкод» на воді

Над внутрішньою відкритою водою (вода за DSWx-S1 на цю дату, ≥ 3 пікселі від берега, постійна за JRC) справжній сигнал L-band нижче −20 дБ. Тому медіана HH/HV там — це **ефективний рівень шуму + перешкод** на дату. Порівнюємо його з NESZ із метаданих і з відбиттям суші.

In [ ]:
def water_core(dswx):
    w = (dswx == 1).astype('f4')
    return (dswx == 1) & (uniform_filter(w, 7) > 0.99) & (np.nan_to_num(JRC, nan=0) >= 80)

QC = []
for t in TRACKS:
    S = STACK[t]
    for i, tt in enumerate(S['time']):
        core = water_core(S['dswx'][i]); m = S['meta'][i]
        r = {'трек': t, 'дата': tt.strftime('%Y-%m-%d'), 'версія': m['version'], 'рівень': S['scenes'].iloc[i]['рівень'],
             'DSWx Δt, діб': round(S['dswx_dt'][i], 1), 'вода-зонд, км²': round(core.sum() * PIX_KM2, 1)}
        for tag in 'BA':
            for pol in ('hh', 'hv'):
                a = S[f'{pol}{tag}'][i]
                r[f'{pol.upper()}{tag} вода'] = np.nanmedian(a[core]) if core.sum() > 50 else np.nan
                r[f'{pol.upper()}{tag} рілля'] = np.nanmedian(a[(WC == 40) & LAND_HIST])
                r[f'NESZ {pol.upper()}{tag}'] = np.nanmedian(S[f'nesz_{pol}{tag}'][i][core]) if core.sum() > 50 else np.nanmedian(S[f'nesz_{pol}{tag}'][i])
            r[f'rfi {tag}'] = m[f'rfi_HH{tag}']
        r['валідних B, %'] = 100 * np.isfinite(S['hhB'][i]).mean()
        QC.append(r)
QC = pd.DataFrame(QC)
show = ['трек', 'дата', 'версія', 'рівень', 'вода-зонд, км²', 'HHB вода', 'HVB вода', 'NESZ HHB', 'HHB рілля', 'HVB рілля', 'HHA вода', 'HVA вода', 'rfi B', 'rfi A', 'валідних B, %']
display(QC[show].round(1))
QC.to_csv(OUT / 'qc_by_date.csv', index=False, encoding='utf-8-sig')

fig, axs = plt.subplots(1, 2, figsize=(15, 4.2), constrained_layout=True, sharey=True)
for ax, tag in zip(axs, 'BA'):
    for t, mk in zip(TRACKS, ('o', 's')):
        q = QC[QC['трек'] == t]; d = pd.to_datetime(q['дата'])
        ax.plot(d, q[f'HH{tag} вода'], mk + '-', c='tab:blue', label=f'{t}: HH над водою')
        ax.plot(d, q[f'HV{tag} вода'], mk + '-', c='tab:red', label=f'{t}: HV над водою')
        ax.plot(d, q[f'HH{tag} рілля'], mk + ':', c='tab:green', label=f'{t}: HH рілля')
        ax.plot(d, q[f'NESZ HH{tag}'], mk + '--', c='gray', lw=.8, label=f'{t}: NESZ HH (метадані)')
    ax.axhspan(-40, -20, color='tab:blue', alpha=.07); ax.text(d.iloc[0], -21, 'тут мала б бути справжня вода', fontsize=8, va='top')
    ax.set_title(f'frequency{tag}: рівень, виміряний на воді, проти суші'); ax.set_ylabel('дБ'); ax.set_ylim(-28, 2)
axs[1].legend(fontsize=7, ncol=2, loc='lower right')
fig.savefig(OUT / 'qc_water_probe.png', dpi=130); plt.show()

## 4. Усі дати: HH і HV frequencyB

Однакова шкала для всіх панелей. Великі розмиті яскраві плями, що змінюються від дати до дати, — перешкоди. Межі полів, русло й дно, що лишаються на місці, — поверхня.

In [ ]:
def small_multiples(t, key, vmin, vmax, title, fname, cmap='gray', data=None):
    S = STACK[t]; arr = S[key] if data is None else data; n = len(arr); nc = 4; nr = int(np.ceil(n / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(4.2 * nc, 3.1 * nr), constrained_layout=True)
    for i, ax in enumerate(axs.ravel()):
        ax.set_axis_off()
        if i < n:
            q = ax.imshow(arr[i][::2, ::2], cmap=cmap, vmin=vmin, vmax=vmax, extent=EXTENT)
            ax.set_title(f'{t} {S["time"][i]:%d.%m}')
    fig.colorbar(q, ax=axs, shrink=.6, label='дБ'); fig.suptitle(title)
    fig.savefig(OUT / fname, dpi=110); plt.show()

for t in TRACKS:
    small_multiples(t, 'hhB', -20, -2, f'HH frequencyB, {t}', f'all_dates_hhB_{t}.png')
    small_multiples(t, 'hvB', -24, -8, f'HV frequencyB, {t}', f'all_dates_hvB_{t}.png')

## 5. Карта «фону перешкод» для кожної дати і експериментальна корекція

Поверхня змінюється в масштабі полів (сотні метрів), а перешкоди тут — розмиті плями на десятки кілометрів. Для кожної дати рахуємо медіану HV і HH **лише на суші** у блоках 2 × 2 км, а потім віднімаємо багатодатну медіану цих блоків. Те, що лишилося, — великомасштабна аномалія цієї дати: в основному перешкоди, але сюди ж потрапить і реальне регіональне зволоження після дощу.

**Корекція** віднімає цю аномалію (у дБ). Це грубе наближення, бо перешкоди додаються в лінійній потужності. Тому нижче перевіряємо, чи корекція справді робить стабільні класи стабільнішими.

In [ ]:
BLK = 25   # 25 × 80 м = 2 км
def block_median(a, mask):
    h, w = (H // BLK) * BLK, (W // BLK) * BLK
    v = np.where(mask, a, np.nan)[:h, :w].reshape(h // BLK, BLK, w // BLK, BLK).transpose(0, 2, 1, 3).reshape(h // BLK, w // BLK, -1)
    out = np.nanmedian(v, -1); out[np.isfinite(v).sum(-1) < BLK * BLK * 0.2] = np.nan
    return out

def upsample(c):
    c = pd.DataFrame(c).interpolate(axis=1, limit_direction='both').interpolate(axis=0, limit_direction='both').to_numpy()
    up = zoom(np.nan_to_num(c, nan=0), BLK, order=1)
    full = np.zeros((H, W), 'f4'); full[:up.shape[0], :up.shape[1]] = up
    full[up.shape[0]:, :] = full[up.shape[0] - 1:up.shape[0], :]; full[:, up.shape[1]:] = full[:, up.shape[1] - 1:up.shape[1]]
    return full

for t in TRACKS:
    S = STACK[t]
    for pol in ('hh', 'hv'):
        bg = np.stack([block_median(S[f'{pol}B'][i], LAND_HIST) for i in range(len(S['time']))])
        anom = bg - np.nanmedian(bg, 0)
        S[f'anom_{pol}'] = anom
        S[f'{pol}C'] = np.stack([S[f'{pol}B'][i] - upsample(anom[i]) for i in range(len(anom))]).astype('f4')
    S['rfi_index'] = [float(np.nanpercentile(np.abs(a), 95)) for a in S['anom_hv']]

for t in TRACKS:
    S = STACK[t]; n = len(S['time']); nc = 6; nr = int(np.ceil(n / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(3.1 * nc, 2.4 * nr), constrained_layout=True)
    for i, ax in enumerate(axs.ravel()):
        ax.set_axis_off()
        if i < n:
            q = ax.imshow(S['anom_hv'][i], cmap='coolwarm', vmin=-5, vmax=5, extent=EXTENT)
            ax.set_title(f'{S["time"][i]:%d.%m}  p95|a|={S["rfi_index"][i]:.1f} дБ')
    fig.colorbar(q, ax=axs, shrink=.7, label='аномалія HV, дБ'); fig.suptitle(f'Великомасштабна аномалія HV (блоки 2 км), {t}')
    fig.savefig(OUT / f'rfi_background_{t}.png', dpi=110); plt.show()

## 6. Що на ділянці стабільне, а що змінюється: багатодатні композити (трек 143_A_025, після корекції)

* **Медіана HH** — найчистіше зображення ділянки: медіана гасить одноденні перешкоди.
* **Мінливість** — стандартне відхилення HH у часі: де поверхня справді живе протягом сезону.
* **Сезонний RGB із HV**: R = червень–липень, G = серпень–вересень, B = жовтень–листопад. Біле/сіре — стабільно; червоне — рослинність була влітку і зникла (збирання врожаю); синє — відбиття зросло восени.

Яскрава пляма в центрі RGB — перешкода, присутня **майже на всіх датах**. Корекція прибирає лише відхилення від багатодатної медіани, тож постійну перешкоду не прибирає: вона сидить у самій медіані.

In [ ]:
def seasonal(t, key, months):
    S = STACK[t]; idx = [i for i, tt in enumerate(S['time']) if tt.month in months]
    return np.nanmedian(S[key][idx], 0) if idx else np.full((H, W), np.nan)

def stretch(a, lo, hi):
    return np.clip((np.nan_to_num(a, nan=lo) - lo) / (hi - lo), 0, 1)

for t in TRACKS:
    S = STACK[t]
    full = np.isfinite(S['hhB']).mean(0) >= 0.9          # прибираємо краї кадру, яких немає на частині дат
    med = np.where(full, np.nanmedian(S['hhC'], 0), np.nan); sd = np.where(full, np.nanstd(S['hhC'], 0), np.nan)
    rgb = np.dstack([stretch(seasonal(t, 'hvC', (6, 7)), -22, -10), stretch(seasonal(t, 'hvC', (8, 9)), -22, -10), stretch(seasonal(t, 'hvC', (10, 11)), -22, -10)]) * full[..., None]
    S['median_hh'], S['std_hh'] = med, sd
    fig, axs = plt.subplots(1, 3, figsize=(18, 5.2), constrained_layout=True)
    q = axs[0].imshow(med, cmap='gray', vmin=-18, vmax=-2, extent=EXTENT); fig.colorbar(q, ax=axs[0], shrink=.8, label='дБ'); axs[0].set_title(f'{t}: медіана HH за всі дати')
    q = axs[1].imshow(median_filter(np.nan_to_num(sd), 3), cmap='magma', vmin=0, vmax=4, extent=EXTENT); fig.colorbar(q, ax=axs[1], shrink=.8, label='σ, дБ'); axs[1].set_title('мінливість HH у часі')
    axs[2].imshow(rgb, extent=EXTENT); axs[2].set_title('HV: R = черв–лип, G = серп–вер, B = жовт–лист')
    fig.savefig(OUT / f'composites_{t}.png', dpi=130); plt.show()

## 7. Часові ряди за типами поверхні і погода

Медіана за класом для кожної дати. **Забудова** — контроль: вона не повинна «гуляти». Поруч — опади й вологість ґрунту ERA5 (Open-Meteo, потрібен інтернет).

Графіки — за **сирими** даними. Після корекції з розділу 5 рілля (більшість суші) за побудовою стає пласкою, бо корекція вирівнює саме фон суші. Скориговані ряди годяться лише для перевірки стабільності забудови.

In [ ]:
ZONES = {'рілля': (WC == 40) & LAND_HIST, 'забудова': (WC == 50) & LAND_HIST, 'трави (поза дном)': (WC == 30) & ~RES_BED,
         'дно: дерева': RES_BED & (WC == 10), 'дно: трави/чагарники/болото': RES_BED & np.isin(WC, (20, 30, 90)),
         'дно: усе': RES_BED}
TS = []
for t in TRACKS:
    S = STACK[t]
    for i, tt in enumerate(S['time']):
        for z, m in ZONES.items():
            TS.append({'трек': t, 'час': tt, 'зона': z, **{f'{k}': float(np.nanmedian(S[k][i][m])) for k in ('hhB', 'hvB', 'hhC', 'hvC')}})
TS = pd.DataFrame(TS)

stab = TS[TS['зона'] == 'забудова'].groupby('трек')[['hhB', 'hhC', 'hvB', 'hvC']].std().round(2)
stab.columns = ['σ HH сирий', 'σ HH кориг.', 'σ HV сирий', 'σ HV кориг.']
print('Розкид медіани забудови між датами (менше — краще):'); display(stab)

WEATHER = None
try:
    pts = [(46.75, 33.37), (47.57, 34.40), (47.84, 35.14), (47.40, 34.50)]
    t_all = sorted(tt for t in TRACKS for tt in STACK[t]['time'])
    q = {'latitude': ','.join(str(p[0]) for p in pts), 'longitude': ','.join(str(p[1]) for p in pts),
         'start_date': (t_all[0] - timedelta(days=10)).date().isoformat(), 'end_date': t_all[-1].date().isoformat(),
         'daily': 'precipitation_sum', 'hourly': 'soil_moisture_0_to_7cm', 'timezone': 'UTC'}
    with urllib.request.urlopen('https://archive-api.open-meteo.com/v1/archive?' + urllib.parse.urlencode(q), timeout=60) as r:
        res = json.load(r)
    res = res if isinstance(res, list) else [res]
    daily = pd.concat([pd.DataFrame(j['daily']).set_index('time') for j in res], axis=1).mean(axis=1).rename('опади, мм')
    hourly = pd.concat([pd.DataFrame(j['hourly']).set_index('time') for j in res], axis=1).mean(axis=1).rename('вологість 0–7 см')
    daily.index, hourly.index = pd.to_datetime(daily.index), pd.to_datetime(hourly.index)
    WEATHER = (daily, hourly)
    TS['вологість'] = [hourly.iloc[np.argmin(np.abs((hourly.index - tt).total_seconds()))] for tt in TS['час']]
except Exception as e:
    print('Погоду не отримано:', e)

fig, axs = plt.subplots(3, 1, figsize=(14, 10), sharex=True, constrained_layout=True)
cols = dict(zip(ZONES, plt.cm.tab10.colors))
for ax, key, lab in ((axs[0], 'hhB', 'HH сирий, дБ'), (axs[1], 'hvB', 'HV сирий, дБ')):
    for z in ZONES:
        for t, ls in zip(TRACKS, ('-', '--')):
            q = TS[(TS['зона'] == z) & (TS['трек'] == t)]
            ax.plot(q['час'], q[key], ls, marker='o' if ls == '-' else 's', ms=4, c=cols[z], label=f'{z} ({t[:5]})' if True else None)
    ax.set_ylabel(lab)
axs[0].legend(fontsize=7, ncol=4); axs[0].set_title('Медіана за зонами (сирі дані): суцільна — 143_A (02:39 UTC), пунктир — 108_D (16:46 UTC)')
if WEATHER is not None:
    axs[2].bar(WEATHER[0].index, WEATHER[0].values, color='tab:blue', width=1, label='опади, мм/добу')
    ax2 = axs[2].twinx(); ax2.plot(WEATHER[1].index, WEATHER[1].values, c='tab:brown', lw=1, label='вологість ґрунту, м³/м³'); ax2.set_ylabel('м³/м³')
    axs[2].set_ylabel('мм/добу'); axs[2].set_title('ERA5: опади (стовпчики) і вологість ґрунту 0–7 см (лінія), середнє 4 точок')
fig.savefig(OUT / 'timeseries_by_zone.png', dpi=130); plt.show()

if 'вологість' in TS:
    rows = []
    for t in TRACKS:
        for z in ('рілля', 'трави (поза дном)', 'забудова'):
            q = TS[(TS['трек'] == t) & (TS['зона'] == z)]
            for k in ('hhB', 'hvB'):
                if len(q) >= 4:
                    r, p = pearsonr(q['вологість'], q[k]); rows.append({'трек': t, 'зона': z, 'ряд': k, 'r з вологістю': r, 'p': p, 'n': len(q)})
    SM_CORR = pd.DataFrame(rows); print('Кореляція медіани HH з вологістю ґрунту ERA5 на момент знімка:'); display(SM_CORR.round(3))

## 8. Вода на кожну дату: NISAR проти DSWx-S1

Класифікатор (градієнтний бустинг; ознаки HH, HV, HH−HV, текстура, скориговані HH/HV) для **кожної дати навчається на всіх інших датах** того самого треку і перевіряється на цій. Так метод ніколи не бачить відповіді для дати, яку оцінюємо. Площу води рахуємо на дні водосховища, лише якщо DSWx покриває ≥ 95 % дна.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

def feats(S, i):
    hh, hv = np.nan_to_num(S['hhB'][i], nan=-30), np.nan_to_num(S['hvB'][i], nan=-30)
    lin = 10 ** (hh / 10); mu = uniform_filter(lin, 5)
    cv = np.sqrt(np.maximum(uniform_filter(lin**2, 5) - mu**2, 0)) / np.maximum(mu, 1e-6)
    return np.stack([median_filter(hh, 3), median_filter(hv, 3), median_filter(hh - hv, 3), cv,
                     median_filter(np.nan_to_num(S['hhC'][i], nan=-30), 3), median_filter(np.nan_to_num(S['hvC'][i], nan=-30), 3)], -1).reshape(-1, 6)

rng = np.random.default_rng(0)
WATER_TS = []
for t in TRACKS:
    S = STACK[t]; n = len(S['time']); samples = []
    valid = [np.isfinite(S['hhB'][i]) & (S['dswx'][i] < 250) for i in range(n)]
    truth = [np.isin(S['dswx'][i], (1, 3)) for i in range(n)]
    for i in range(n):
        F = feats(S, i); idx = np.flatnonzero(valid[i]); wv = truth[i].ravel()[idx]
        sel = np.r_[rng.choice(idx[wv], min(15000, wv.sum()), replace=False), rng.choice(idx[~wv], min(45000, (~wv).sum()), replace=False)]
        samples.append((F[sel], truth[i].ravel()[sel]))
    S['water_pred'] = np.zeros((n, H, W), bool)
    for i in range(n):
        Xtr = np.concatenate([s[0] for j, s in enumerate(samples) if j != i]); ytr = np.concatenate([s[1] for j, s in enumerate(samples) if j != i])
        clf = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(Xtr, ytr)
        pred = clf.predict_proba(feats(S, i))[:, 1].reshape(H, W) > 0.5
        S['water_pred'][i] = pred
        m = valid[i]; p, w = pred[m], truth[i][m]
        tp, fp, fn = (p & w).sum(), (p & ~w).sum(), (~p & w).sum()
        bed_cov = (S['dswx'][i][RES_BED] < 250).mean()
        WATER_TS.append({'трек': t, 'час': S['time'][i], 'IoU': tp / (tp + fp + fn), 'рівень на воді HV B': QC[(QC['трек'] == t)].iloc[i]['HVB вода'],
                         'rfi_index': S['rfi_index'][i],
                         'вода DSWx на дні, км²': (truth[i] & RES_BED).sum() * PIX_KM2 if bed_cov >= .95 else np.nan,
                         'вода NISAR на дні, км²': (pred & RES_BED & np.isfinite(S['hhB'][i])).sum() * PIX_KM2 if bed_cov >= .95 else np.nan})
WATER_TS = pd.DataFrame(WATER_TS); display(WATER_TS.round({'IoU': 3, 'рівень на воді HV B': 1, 'rfi_index': 2, 'вода DSWx на дні, км²': 1, 'вода NISAR на дні, км²': 1}))

fig, axs = plt.subplots(1, 2, figsize=(15, 4.3), constrained_layout=True)
for t, mk in zip(TRACKS, 'os'):
    q = WATER_TS[WATER_TS['трек'] == t]
    axs[0].plot(q['час'], q['вода DSWx на дні, км²'], mk + '-', c='tab:blue', label=f'DSWx-S1 ({t[:5]} дати)')
    axs[0].plot(q['час'], q['вода NISAR на дні, км²'], mk + '--', c='tab:orange', label=f'NISAR ({t[:5]})')
    axs[1].scatter(q['рівень на воді HV B'], q['IoU'], marker=mk, label=t)
axs[0].set_title('Площа води на дні колишнього водосховища'); axs[0].set_ylabel('км²'); axs[0].legend(fontsize=8)
axs[1].set(xlabel='HV B над водою (рівень перешкод), дБ', ylabel='IoU з DSWx-S1', title='Чим вищі перешкоди, тим гірше NISAR бачить воду?'); axs[1].legend()
fig.savefig(OUT / 'water_timeseries.png', dpi=130); plt.show()
_q = WATER_TS.dropna(subset=['рівень на воді HV B', 'IoU']); r_iou = pearsonr(_q['рівень на воді HV B'], _q['IoU'])
print(f'Кореляція IoU з рівнем перешкод на воді: r = {r_iou[0]:.2f} (p = {r_iou[1]:.3f})')

# Частота води за сезон: як часто піксель був водою
fig, axs = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
S = STACK[MAIN]; vd = (S['dswx'] < 250)
fr_dswx = np.where(vd.sum(0) > 0, (np.isin(S['dswx'], (1, 3)) & vd).sum(0) / np.maximum(vd.sum(0), 1), np.nan)
fr_nis = (S['water_pred'] & np.isfinite(S['hhB'])).sum(0) / np.maximum(np.isfinite(S['hhB']).sum(0), 1)
for ax, a, ttl in ((axs[0], fr_dswx, 'DSWx-S1: частка дат, коли піксель — вода'), (axs[1], fr_nis, f'NISAR {MAIN}: частка дат, коли піксель — вода')):
    q = ax.imshow(a, cmap='Blues', vmin=0, vmax=1, extent=EXTENT); ax.set_title(ttl); fig.colorbar(q, ax=ax, shrink=.8)
fig.savefig(OUT / 'water_frequency.png', dpi=130); plt.show()
ok = np.isfinite(fr_dswx) & RES_BED
print(f'Збіг частоти води на дні (кореляція пікселів): r = {np.corrcoef(fr_dswx[ok], fr_nis[ok])[0, 1]:.2f}')

## 9. Сезонні зміни на ріллі: коли зникла рослинність (збирання врожаю)

Для кожного пікселя ріллі знаходимо проміжок між сусідніми датами з **найбільшим падінням HV** (рослинність деполяризує, тож після збирання HV різко падає). Падіння < 3 дБ не рахуємо.

**Фільтри від перешкод:** (1) відкидаємо проміжки, де рівень перешкод на воді стрибнув більше ніж на 3 дБ (тоді «падіння» на всій ділянці дає перешкода); (2) рахуємо лише пікселі, де HV на обох датах вищий за рівень перешкод на воді. Без фільтрів піки гістограми припадають саме на проміжки зі стрибками перешкод.

**Реальність для звірки:** календар збирання на півдні України: озима пшениця і ячмінь — кінець червня–липень, ріпак — липень, соняшник — кінець серпня–вересень, кукурудза — вересень–жовтень.

**Перевірка перешкодами:** треки 143_A і 108_D літають у різний час доби, і перешкоди в них різні. Якщо дата падіння в пікселі збігається в обох треках частіше, ніж випадково, це реальна подія на полі.

In [ ]:
def floor_series(t, col):
    """Рівень перешкод на воді для кожної дати; пропуски заповнюємо інтерполяцією в часі."""
    q = QC[QC['трек'] == t][col].astype(float).reset_index(drop=True)
    return q.interpolate(limit_direction='both').to_numpy()

def drop_date(t, key='hvB', thr=-3, max_step=3.0, snr=0.0):
    """Найбільше падіння між сусідніми датами. Відкидаємо проміжки, де рівень перешкод на воді стрибнув
    більше ніж на max_step дБ, і пікселі, де сигнал на обох датах не вищий за цей рівень + snr."""
    S = STACK[t]; fin = np.isfinite(S[key]); fl = floor_series(t, f'{key[:2].upper()}B вода')
    a = np.stack([median_filter(np.nan_to_num(x, nan=-30), 3) for x in S[key]])
    above = a > fl[:, None, None] + snr
    good_iv = np.abs(np.diff(fl)) <= max_step
    d = np.diff(a, axis=0)
    d = np.where(fin[1:] & fin[:-1] & above[1:] & above[:-1] & good_iv[:, None, None], d, 0)
    k = np.argmin(d, 0); mn = np.take_along_axis(d, k[None], 0)[0]
    mid = np.array([S['time'][i] + (S['time'][i + 1] - S['time'][i]) / 2 for i in range(len(S['time']) - 1)])
    doy = np.array([m.timetuple().tm_yday for m in mid], 'f4')[k]
    ok = (mn < thr) & (WC == 40) & LAND_HIST
    return np.where(ok, doy, np.nan), mid, good_iv

CROP = (WC == 40) & LAND_HIST
for t in TRACKS:
    fl = floor_series(t, 'HVB вода'); S = STACK[t]
    iv = pd.DataFrame({'проміжок': [f"{S['time'][i]:%d.%m}–{S['time'][i + 1]:%d.%m}" for i in range(len(fl) - 1)],
                       'стрибок перешкод HV, дБ': np.diff(fl)})
    iv['надійний'] = iv['стрибок перешкод HV, дБ'].abs() <= 3
    iv['рілля вище рівня перешкод на обох датах, %'] = [
        100 * np.mean(((np.nan_to_num(S['hvB'][i], nan=-30) > fl[i]) & (np.nan_to_num(S['hvB'][i + 1], nan=-30) > fl[i + 1]))[CROP])
        for i in range(len(fl) - 1)]
    print(t); display(iv.round(1))

DROP = {t: drop_date(t) for t in TRACKS}
DROP_RAW = {t: drop_date(t, max_step=99, snr=-99) for t in TRACKS}   # без фільтрів, для порівняння
fig, axs = plt.subplots(1, 3, figsize=(19, 5.2), constrained_layout=True)
for ax, t in zip(axs[:2], TRACKS):
    q = ax.imshow(DROP[t][0], cmap='turbo', vmin=170, vmax=310, extent=EXTENT, interpolation='nearest')
    ax.set_title(f'{t}: день року найбільшого падіння HV на ріллі'); cb = fig.colorbar(q, ax=ax, shrink=.8)
    cb.set_ticks([182, 213, 244, 274, 305]); cb.set_ticklabels(['1 лип', '1 серп', '1 вер', '1 жовт', '1 лист'])
for t, c in zip(TRACKS, ('tab:blue', 'tab:orange')):
    v = DROP[t][0][np.isfinite(DROP[t][0])]; v0 = DROP_RAW[t][0][np.isfinite(DROP_RAW[t][0])]
    axs[2].hist(v0, bins=np.arange(170, 312, 6), histtype='step', color=c, ls=':', label=f'{t} без фільтрів: {v0.size * PIX_KM2:,.0f} км²')
    axs[2].hist(v, bins=np.arange(170, 312, 6), alpha=.55, color=c, label=f'{t} з фільтрами: {v.size * PIX_KM2:,.0f} км²')
for lo, hi, name in ((176, 212, 'пшениця/ячмінь'), (235, 273, 'соняшник'), (258, 300, 'кукурудза')):
    axs[2].axvspan(lo, hi, alpha=.08, color='k'); axs[2].text((lo + hi) / 2, axs[2].get_ylim()[1] * .9, name, ha='center', fontsize=8)
axs[2].set(xlabel='день року', ylabel='пікселів', title='Коли падало HV: два треки проти календаря збирання'); axs[2].legend(fontsize=8)
fig.savefig(OUT / 'harvest_timing.png', dpi=130); plt.show()

a, b = DROP[MAIN][0], DROP[ALT][0]; both = np.isfinite(a) & np.isfinite(b)
share = np.mean(np.abs(a[both] - b[both]) <= 12) if both.any() else np.nan
shuf = np.mean(np.abs(a[both] - rng.permutation(b[both])) <= 12) if both.any() else np.nan
print(f'З фільтрами: рілля з падінням в обох треках: {both.sum() * PIX_KM2:,.0f} км². Дати падіння збігаються (±12 діб) у {share:.0%} пікселів; '
      f'випадковий збіг (перемішування) {shuf:.0%}.')
for t in TRACKS:
    v = DROP[t][0][np.isfinite(DROP[t][0])]
    by = pd.Series(pd.cut(v, [0, 213, 244, 274, 400], labels=['до 1 серп', 'серпень', 'вересень', 'жовтень+'])).value_counts(normalize=True).sort_index()
    print(t, f'з фільтрами {v.size * PIX_KM2:,.0f} км² ріллі; частки падінь:', ', '.join(f'{k} {x:.0%}' for k, x in by.items()))
a0, b0 = DROP_RAW[MAIN][0], DROP_RAW[ALT][0]; both0 = np.isfinite(a0) & np.isfinite(b0)
share0 = np.mean(np.abs(a0[both0] - b0[both0]) <= 12) if both0.any() else np.nan
print(f'Без фільтрів: збіг дат між треками {share0:.0%} на {both0.sum() * PIX_KM2:,.0f} км²')

## 10. Стійкі зміни «початок → кінець сезону» і перевірка другим треком

Медіана перших трьох дат проти медіани останніх трьох (окремо для кожного треку). Медіана трьох дат гасить одноденні перешкоди. Якщо карти змін двох **незалежних** треків корелюють, зміни реальні.

In [ ]:
PERSIST = {}
for t in TRACKS:
    S = STACK[t]
    for key in ('hhC', 'hvC'):
        PERSIST[(t, key)] = median_filter(np.nan_to_num(np.nanmedian(S[key][-3:], 0) - np.nanmedian(S[key][:3], 0), nan=0), 3)
    print(t, 'початок:', ', '.join(f'{x:%d.%m}' for x in S['time'][:3]), '| кінець:', ', '.join(f'{x:%d.%m}' for x in S['time'][-3:]))

fig, axs = plt.subplots(2, 2, figsize=(15, 9.5), constrained_layout=True)
for i, key in enumerate(('hhC', 'hvC')):
    for j, t in enumerate(TRACKS):
        q = axs[i, j].imshow(PERSIST[(t, key)], cmap='coolwarm', vmin=-6, vmax=6, extent=EXTENT)
        axs[i, j].set_title(f'{t}: Δ {key[:2].upper()} (кінець − початок сезону)'); fig.colorbar(q, ax=axs[i, j], shrink=.8, label='Δ дБ')
fig.savefig(OUT / 'persistent_change.png', dpi=130); plt.show()

# Карта, де обидва треки незалежно бачать одну й ту саму стійку зміну HH (> 2 дБ одного знаку)
agree = np.select([(PERSIST[(MAIN, 'hhC')] > 2) & (PERSIST[(ALT, 'hhC')] > 2), (PERSIST[(MAIN, 'hhC')] < -2) & (PERSIST[(ALT, 'hhC')] < -2)], [2, 1], 0)
fig, ax = plt.subplots(figsize=(9, 6.5), constrained_layout=True)
ax.imshow(np.where(np.isfinite(STACK[MAIN]['median_hh']), STACK[MAIN]['median_hh'], np.nan), cmap='gray', vmin=-18, vmax=-2, extent=EXTENT)
ax.imshow(np.ma.masked_equal(agree, 0), cmap=ListedColormap(['#2b6cff', '#ff3b2b']), vmin=1, vmax=2, extent=EXTENT, interpolation='nearest')
ax.set_title('Обидва треки: HH кінця сезону нижче (синє) / вище (червоне) за початок більш ніж на 2 дБ')
fig.savefig(OUT / 'cross_track_agreement_map.png', dpi=130); plt.show()
print(f"Узгоджене зниження: {(agree == 1).sum() * PIX_KM2:,.0f} км², узгоджене зростання: {(agree == 2).sum() * PIX_KM2:,.0f} км²; "
      f"з них на дні водосховища {((agree > 0) & RES_BED).sum() * PIX_KM2:,.0f} км², на ріллі {((agree > 0) & ZONES['рілля']).sum() * PIX_KM2:,.0f} км²")

rows = []
valid_both = np.isfinite(STACK[MAIN]['hhB']).all(0) & np.isfinite(STACK[ALT]['hhB']).all(0)
# Зона плями перешкод: де багатодатний HV-фон суші в будь-якому треку вищий за медіану по AOI більш ніж на 3 дБ
_bg = {t: upsample(np.nanmedian(np.stack([block_median(x, LAND_HIST) for x in STACK[t]['hvB']]), 0)) for t in TRACKS}
RFI_ZONE = np.zeros((H, W), bool)
for t in TRACKS:
    RFI_ZONE |= _bg[t] > np.nanmedian(_bg[t][LAND_HIST]) + 3
print(f'Зона постійної плями перешкод: {RFI_ZONE.mean():.0%} AOI')
for z, m in {'рілля': ZONES['рілля'], 'рілля поза плямою перешкод': ZONES['рілля'] & ~RFI_ZONE, 'дно: усе': RES_BED,
             'дно поза плямою перешкод': RES_BED & ~RFI_ZONE, 'уся AOI': np.ones((H, W), bool)}.items():
    for key in ('hhC', 'hvC'):
        mm = m & valid_both; a, b = PERSIST[(MAIN, key)][mm], PERSIST[(ALT, key)][mm]
        rows.append({'зона': z, 'поляр.': key[:2].upper(), 'км²': mm.sum() * PIX_KM2, 'r між треками': np.corrcoef(a, b)[0, 1],
                     f'медіана Δ {MAIN}': np.median(a), f'медіана Δ {ALT}': np.median(b),
                     'частка, де обидва |Δ|>3 дБ і одного знаку, %': 100 * np.mean((np.abs(a) > 3) & (np.abs(b) > 3) & (np.sign(a) == np.sign(b)))})
XTRACK = pd.DataFrame(rows); display(XTRACK.round(3))

h = PERSIST[(MAIN, 'hhC')]; g = PERSIST[(ALT, 'hhC')]; mm = ZONES['рілля'] & valid_both & ~RFI_ZONE
idx = rng.choice(np.flatnonzero(mm), min(60000, mm.sum()), replace=False)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.hexbin(h.ravel()[idx], g.ravel()[idx], gridsize=60, bins='log', extent=(-10, 10, -10, 10), mincnt=1)
ax.plot([-10, 10], [-10, 10], 'k--', lw=.7); ax.set(xlabel=f'Δ HH {MAIN}, дБ', ylabel=f'Δ HH {ALT}, дБ', title='Рілля поза плямою: стійка зміна HH у двох треках')
fig.savefig(OUT / 'cross_track_scatter.png', dpi=130); plt.show()

## 11. Підсумок у числах

In [ ]:
L = []
for t in TRACKS:
    q = QC[QC['трек'] == t]
    L.append(f"{t}: {len(q)} дат ({q['дата'].iloc[0]} … {q['дата'].iloc[-1]}). HH B над водою: медіана {q['HHB вода'].median():.1f} дБ "
             f"(від {q['HHB вода'].min():.1f} до {q['HHB вода'].max():.1f}); NESZ із метаданих {q['NESZ HHB'].median():.1f} дБ. "
             f"HV A над водою {q['HVA вода'].median():.1f} дБ.")
L.append('Розкид медіани забудови між датами (σ HH сирий → кориг.): ' + '; '.join(f"{t}: {stab.loc[t, 'σ HH сирий']:.2f} → {stab.loc[t, 'σ HH кориг.']:.2f} дБ" for t in TRACKS))
L.append('Вода NISAR проти DSWx-S1 (IoU, навчання на інших датах): ' + '; '.join(f"{t}: медіана {WATER_TS[WATER_TS['трек'] == t]['IoU'].median():.2f}, найкраща {WATER_TS[WATER_TS['трек'] == t]['IoU'].max():.2f}" for t in TRACKS)
         + f". Кореляція IoU з рівнем перешкод r = {r_iou[0]:.2f}.")
L.append(f'Частота води на дні за сезон, NISAR проти DSWx: r = {np.corrcoef(fr_dswx[ok], fr_nis[ok])[0, 1]:.2f}.')
L.append(f'Збирання врожаю (падіння HV ≥ 3 дБ, з фільтрами від перешкод): {np.isfinite(DROP[MAIN][0]).sum() * PIX_KM2:,.0f} / {np.isfinite(DROP[ALT][0]).sum() * PIX_KM2:,.0f} км² ріллі; '
         f'дати в двох треках збігаються в {share:.0%} пікселів ({both.sum() * PIX_KM2:,.0f} км²) проти {shuf:.0%} випадково.')
for _, r in XTRACK.iterrows():
    L.append(f"Стійка зміна {r['поляр.']} ({r['зона']}): кореляція між треками r = {r['r між треками']:.2f}, медіани {r[f'медіана Δ {MAIN}']:+.1f} / {r[f'медіана Δ {ALT}']:+.1f} дБ.")
if 'вологість' in TS:
    c = SM_CORR[(SM_CORR['зона'] == 'рілля') & (SM_CORR['ряд'] == 'hhB')]
    L.append('Рілля: кореляція HH (сирий) з вологістю ґрунту ERA5: ' + '; '.join(f"{r['трек']}: r = {r['r з вологістю']:.2f}" for _, r in c.iterrows()))
print('\n'.join(L))
(OUT / 'summary.txt').write_text('\n'.join(L), encoding='utf-8')
for name, df in (('timeseries_by_zone', TS), ('water_by_date', WATER_TS), ('cross_track', XTRACK)):
    df.to_csv(OUT / f'{name}.csv', index=False, encoding='utf-8-sig')
print('\nУсе збережено в', OUT)

## 12. Висновки (22 знімки, два треки, червень–листопад 2026)

**Що надійно бачить NISAR на цій ділянці**
1. **Сезонні зміни на рівні цілих полів.** Стійка зміна «початок → кінець сезону» корелює між двома незалежними треками (r ≈ 0.55 для HH), **і поза плямою перешкод так само** (r ≈ 0.55). На ~340 км² ріллі обидва треки показують ту саму зміну > 2 дБ, і вона повторює межі полів: збирання, оранка, сходи озимих. Це реальні зміни, бо перешкоди в двох треках різні.
2. **Вода в динаміці сезону.** Частота води на дні за сезон у NISAR корелює з DSWx-S1 (r ≈ 0.6). Уздовж русла обидва треки показують зниження HH наприкінці сезону. DSWx-S1 водночас фіксує більше води на дні в жовтні–листопаді (до ~360 км² проти ~280 улітку).
3. **Дно колишнього водосховища поза плямою:** HV зріс приблизно на +1 дБ в обох треках. Це узгоджується з густішанням молодої рослинності на дні, але площа перевірки мала (~50 км²).

**Чого NISAR тут не бачить і чому**
4. **Рівень перешкод на воді** вищий за NESZ із метаданих на ~10 дБ і стрибає від дати до дати на 4–17 дБ. У 108_D (вечірній прохід) перешкоди сильніші, ніж у 143_A (нічний). frequencyA непридатна в обох треках: вода там яскравіша за сушу.
5. **Вода на окремій даті:** IoU з DSWx-S1 лише 0.1–0.16, і що вищі перешкоди, то він гірший (r ≈ −0.57). На чистих сценах очікується 0.7–0.9.
6. **Дата збирання врожаю з HV:** HV ріллі вищий за рівень перешкод лише на 0–12 % площі (один проміжок у 108_D — 33 %). Без фільтрів піки «збирання» припадають рівно на стрибки перешкод. З фільтрами треки збігаються не частіше за випадковість.
7. **Зв'язок із вологістю ґрунту** на ріллі не видно (r ≈ 0–0.2): ефект вологи тоне в перешкодах.

**Що це означає для проєкту**
* Найкращий метод для цієї ділянки — **багатодатні медіани і перевірка двома треками**. Попарні порівняння дат дають переважно перешкоди.
* Для звірки з реальністю вже працюють DSWx-S1 (вода) і збіг двох треків (поля). Наступний крок — оптика Sentinel-2 / HLS за ясні дні (серпень–вересень), щоб підписати типи змін на полях, і більше треків (036_D_065, 042_A_026, 137_D_064), щоб збільшити кількість незалежних перевірок.
* Ноутбук не робить висновків про джерела перешкод.